# Week 2: Implementation of Data Cleaning Techniques

## Handling Missing Values

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Age': [25, 30, np.nan, 40, 35],
    'Department': ['HR', 'Finance', 'Finance', np.nan, 'IT']
})

df_imputed = df.copy()
df_imputed['Age'].fillna(df_imputed['Age'].mean(), inplace=True)
df_imputed['Department'].fillna(df_imputed['Department'].mode()[0], inplace=True)
print(df_imputed)

    Age Department
0  25.0         HR
1  30.0    Finance
2  32.5    Finance
3  40.0    Finance
4  35.0         IT


C:\Users\aksha\AppData\Local\Temp\ipykernel_31068\107252875.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_imputed['Age'].fillna(df_imputed['Age'].mean(), inplace=True)
C:\Users\aksha\AppData\Local\Temp\ipykernel_31068\107252875.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

### Forward Fill

In [2]:
df_ffill = df.copy()
df_ffill.fillna(method='ffill', inplace=True)
print(df_ffill)

    Age Department
0  25.0         HR
1  30.0    Finance
2  30.0    Finance
3  40.0    Finance
4  35.0         IT


C:\Users\aksha\AppData\Local\Temp\ipykernel_31068\3989740104.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill.fillna(method='ffill', inplace=True)


### Backward Fill

In [3]:
df_bfill = df.copy()
df_bfill.fillna(method='bfill', inplace=True)
print(df_bfill)

    Age Department
0  25.0         HR
1  30.0    Finance
2  40.0    Finance
3  40.0         IT
4  35.0         IT


C:\Users\aksha\AppData\Local\Temp\ipykernel_31068\3163818295.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_bfill.fillna(method='bfill', inplace=True)


### Deletion

In [4]:
df_drop_rows = df.dropna()
print(df_drop_rows)

df_drop_cols = df.dropna(axis=1)
print(df_drop_cols)

    Age Department
0  25.0         HR
1  30.0    Finance
4  35.0         IT
Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


## Removing Duplicates

### Exact Match Removal

In [5]:
import pandas as pd

df_dups = pd.DataFrame({
    'ID': [1, 2, 2, 3, 4, 4],
    'Name': ['Alice', 'Bob', 'Bob', 'Charlie', 'David', 'David'],
    'Age': [25, 30, 30, 35, 40, 40]
})
df_exact = df_dups.drop_duplicates()
print(df_exact)

   ID     Name  Age
0   1    Alice   25
1   2      Bob   30
3   3  Charlie   35
4   4    David   40


### Subset-Based Removal

In [6]:
df_subset_id = df_dups.drop_duplicates(subset=['ID'])
print(df_subset_id)

df_subset_name = df_dups.drop_duplicates(subset=['Name'])
print(df_subset_name)

   ID     Name  Age
0   1    Alice   25
1   2      Bob   30
3   3  Charlie   35
4   4    David   40
   ID     Name  Age
0   1    Alice   25
1   2      Bob   30
3   3  Charlie   35
4   4    David   40


## Correcting Inconsistent Formats

### Date Standardization

In [7]:
import pandas as pd

df_dates = pd.DataFrame({
    'Date': ['2025-01-05', '05/01/2025', 'Jan 5, 2025', '2025.01.05']
})
df_dates['Date'] = pd.to_datetime(df_dates['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
print(df_dates)

         Date
0  2025-01-05
1         NaN
2         NaN
3         NaN


### Case Normalization

In [8]:
df_text = pd.DataFrame({
    'Name': ['Alice', 'BOB', 'charlie', 'DAVID']
})
df_text['Name_lower'] = df_text['Name'].str.lower()
df_text['Name_upper'] = df_text['Name'].str.upper()
print(df_text)

      Name Name_lower Name_upper
0    Alice      alice      ALICE
1      BOB        bob        BOB
2  charlie    charlie    CHARLIE
3    DAVID      david      DAVID


## Detecting and Treating Outliers

### Z-Score and IQR Detection

In [9]:
import numpy as np
import pandas as pd

np.random.seed(42)
data = np.random.normal(loc=50, scale=5, size=100)
data = np.append(data, [10, 85, 90, 95])
df_outliers = pd.DataFrame({'Value': data})

mean = df_outliers['Value'].mean()
std = df_outliers['Value'].std()
df_outliers['Z_Score'] = (df_outliers['Value'] - mean) / std
z_outliers = df_outliers[np.abs(df_outliers['Z_Score']) > 3]
print(z_outliers)

Q1 = df_outliers['Value'].quantile(0.25)
Q3 = df_outliers['Value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
iqr_outliers = df_outliers[(df_outliers['Value'] < lower_bound) | (df_outliers['Value'] > upper_bound)]
print(iqr_outliers)

     Value   Z_Score
100   10.0 -4.430142
101   85.0  3.820687
102   90.0  4.370743
103   95.0  4.920798
         Value   Z_Score
74   36.901274 -1.470704
100  10.000000 -4.430142
101  85.000000  3.820687
102  90.000000  4.370743
103  95.000000  4.920798


### Outlier Treatment

In [10]:
df_capped = df_outliers.copy()
df_capped['Value_Capped'] = np.clip(df_capped['Value'], lower_bound, upper_bound)
print(df_capped['Value_Capped'].min(), df_capped['Value_Capped'].max())

38.584249725808036 61.014174850367255
